# Interactive Memory Dump Analyzer

memory.vmem 파일을 로드해 분석 결과를 확인하고, LLM에 자유롭게 질문할 수 있는 웹 기반 노트북입니다.

**사용 순서**: 셀을 순서대로 실행 → 마지막 셀 채팅 UI에서 질문 입력

In [1]:
import sys, os, json
from pathlib import Path
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / 'configs' / 'jupyter_ai_openrouter.env')

DUMP_PATH = os.environ.get('DUMP_PATH', '/home/taejin/Jupyter/data/memory/memory.vmem')
MODEL = os.environ.get('OPENAI_MODEL', 'nvidia/nemotron-3-super-120b-a12b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')

print(f'dump   : {DUMP_PATH}')
print(f'exists : {Path(DUMP_PATH).exists()}')
print(f'model  : {MODEL}')
print(f'api key: {"OK" if os.environ.get("OPENAI_API_KEY") else "MISSING"}')

dump   : /home/taejin/Jupyter/data/memory/memory.vmem
exists : True
model  : nvidia/nemotron-3-super-120b-a12b:free
api key: OK


## Step 1. 분석 컨텍스트 로드

memory.vmem에서 패턴/문자열/프로세스/네트워크 정보를 추출합니다. (수십 초 소요)

In [2]:
from memory_kernel_analyzer import build_analysis_context, summarize_findings
from llm_assistant import LLMAssistant

print('분석 중... (4GB 파일 샘플링 중)')
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('완료.')

분석 중... (4GB 파일 샘플링 중)


완료.


## Step 2. 분석 결과 요약

In [3]:
import ipywidgets as widgets
from IPython.display import display, HTML

fi = summary['file_info']
ns = summary['network_summary']
ps = summary['process_summary']

html = f"""
<style>
  .card {{ background:#1e1e2e; border-radius:8px; padding:16px; margin:8px 0; color:#cdd6f4; font-family:monospace; }}
  .card h3 {{ color:#89b4fa; margin:0 0 10px 0; font-size:14px; }}
  .tag {{ display:inline-block; background:#313244; border-radius:4px; padding:2px 8px; margin:2px; font-size:12px; }}
  .warn {{ color:#f38ba8; }} .ok {{ color:#a6e3a1; }} .info {{ color:#fab387; }}
</style>
<div class="card">
  <h3>파일 정보</h3>
  <span class="tag">{fi['path'].split('/')[-1]}</span>
  <span class="tag info">{fi['size_gb']:.1f} GB</span>
  <span class="tag ok">{summary['kernel_versions'][0] if summary['kernel_versions'] else 'unknown'}</span>
</div>
<div class="card">
  <h3>패닉 패턴 <span class="warn">({summary['panic_pattern_count']}건)</span></h3>
  {''.join(f'<span class="tag warn">{p}</span>' for p in summary['panic_patterns'])}
</div>
<div class="card">
  <h3>주요 문자열</h3>
  {''.join(f'<span class="tag">{s[:60]}</span>' for s in summary['interesting_strings_preview'][:6])}
</div>
<div class="card">
  <h3>네트워크</h3>
  <span class="tag">내부 IP: {ns['internal_ip_count']}</span>
  <span class="tag warn">외부 IP: {ns['external_ip_count']}</span>
  {''.join(f'<span class="tag info">{p}</span>' for p in ns['interesting_ports'][:4])}
</div>
<div class="card">
  <h3>프로세스</h3>
  <span class="tag">{ps['process_count']} 프로세스</span>
  {''.join(f'<span class="tag ok">{u}</span>' for u in ps['users'])}
</div>
"""
display(HTML(html))

## Step 3. LLM 초기 분석

In [ ]:
# %%ai 매직 로드 (선택적 사용 — LLMAssistant 방식과 병행 가능)
%load_ext jupyter_ai_magics

In [4]:
from IPython.display import Markdown

print('LLM 분석 요청 중...')
initial_analysis = assistant.analyze_findings(
    {k: v for k, v in summary.items() if k != 'raw_chunk_samples'},
    task='이 memory dump의 핵심 이상 징후와 root cause 후보를 분석해줘. 신뢰도 기준으로 표로 정리해줘.'
)
display(Markdown(initial_analysis))

LLM 분석 요청 중...


**메모리 덤프 이상 징후 및 Root‑Cause 후보 분석**  
*(vmlinux 심볼 없음 → 문자열/패턴 기반 feasibility 분석)*  

| # | 이상 징후 (Observed Artifact) | 설명 / 의미 | 신뢰도* | 추정되는 Root‑Cause | 근거 / 증거 |
|---|------------------------------|-------------|--------|----------------------|-------------|
| 1 | **`alloc magic is broken at %p: %lx`** | 커널 슬랩/페이지 할당자에서 매직 값이 손상됨을 나타내는 디버그 메시지. 메타데이터가 overwritten됨 → 메모리 손상. | **High** | 메모리 손상 (불량 RAM, 과열, 혹은 커널 버그에 의한 오버런) | 문자열 직접 존재; panic 패턴 `kernel_oops:BUG:`와 자주 함께 나타남. |
| 2 | **`out of memory`** | 시스템이 가용 메모리를 모두 소모했음을 알리는 메시지. OOM killer가 작동했을 가능성. | **Medium** | 메모리 소모 폭주 (메모리 누수, 프로세스 과다 할당) 또는 초기 할당 실패 (GRUB/BIOS 단계) | 문자열 존재; 프로세스 수 50개 중 하나 이상이 비정상적으로 큰 메모리를 점유했을 가능성. |
| 3 | **GRUB 관련 할당 함수 (`grub_malloc`, `grub_calloc`, `grub_realloc`, `grub_zalloc`)** | 부트로더(GRUB)가 메모리를 할당/해제하는 루틴. 이 문자열이 덤프에 남아 있다는 것은 부팅 단계에서 메모리 할당이 발생했고, 아마도 실패했음을 시사. | **Medium** | 부팅 시 메모리 초기화 오류 (BIOS/UEFI 메모리 맵 오류, 불량 DIMM) | PXE‑E20 BIOS 에러와 함께 나타남; 부팅 초기에 메모리 할당 실패 시 커널이 로드되지 않아 오ops 발생. |
| 4 | **`PXE-E20: BIOS extended memory copy error.  AH =`** | BIOS가 확장 메모리(>1MB)를 복사하는 과정에서 하드웨어 오류가 발생했음을 나타내는 PXE/EFI 오류 메시지. | **High** | 하드웨어 메모리 결함 (불량 DIMM, 메모리 컨트롤러 오류, 전압 불안정) | 문자열 직접 존재; `alloc magic is broken`과 함께 나타나면 하드웨어 손상의 강한 신호. |
| 5 | **`libdconfsettings.so` 경로 문자열** | GSettings 백엔드 라이브러리. 해당 라이브러리가 로드된 프로세스(예: `dconf-service` 또는 GNOME 관련)가 비정상 종료했을 가능성. | **Low** | 사용자 공간 프로세스 크래시 (예: dconf 서비스 오류) → 커널 오ops와는 무관할 수 있음 | 문자열만 존재; 스택 트레이스나 프로세스 이름 없음 → 추정 수준 낮음. |
| 6 | **다수 외부 IP 및 포트 (20,22,23,25,53,443) 기록** | 덤프 내에 네트워크 패킷/소켓 관련 문자열이 다수 존재 → 외부와의 통신 활동이 많았음. | **Low‑Medium** | 네트워크 기반 공격 또는 스캔 (예: 포트 스캔, brute‑force) → 프로세스 비정상 행동 → 메모리 소모/손상 유발 가능성 | `interesting_ports` 목록에 일반적인 서비스 포트 포함; 그러나 프로세스/네트워크 상세 정보 없음 → 확신도 낮음. |
| 7 | **`panic_pattern_count: 2` (kernel_oops:BUG:, crashes:crash)** | 두 가지 유형의 오ops/크래시 패턴이 감지됨 → 커널이 비정상 종료된 시점이 최소 두 번 존재. | **High** | 위의 메모리 손상/OOM/hardware 오류 중 하나 이상이 커널 오ops를 트리거함 | 패턴 자체가 존재; 다른 증거와 연관 지어 해석. |

\* **신뢰도 기준**  
- **High**: 문자열이 직접적으로 커널 오ops와 연결되는 핵심 디버그/에러 메시지이며, 하드웨어/메모리 손상과 강한 연관성이 있음.  
- **Medium**: 문자열이 존재하지만, 정확한 원인 단계를 특정하기엔 보완 정보(스택 트레이스, vmlinux)가 부족함.  
- **Low**: 문자열만 존재하고, 해당 코드/프로세스와의 직접적인 연관성을 입증할 증거가 부족함.

### 종합적인 해석
1. **주요 sospeкт**: **하드웨어 메모리 결함** (불량 DIMM 또는 메모리 컨트롤러 오류).  
   - `alloc magic is broken` + `PXE-E20 BIOS extended memory copy error` 두 가지가 동시에 나타나는 것은 메모리 전송/접근 시 데이터 손상이 발생했음을 강하게 시사함.  
   - 이와 함께 `out of memory`가 나타날 수 있는 이유는 손상된 메모리 영역이 할당자에서 사용 불가로 판단되어 가용 메모리가 급감했기 때문일 수 있음.

2. **보조 가능성**: **커널 버그 또는 드라이버 오류**가 메모리 할당 경로에서 오버런을 일으켜 슬랩 매직 값을 훼손시켰을 가능성.  
   - vmlinux 없이 정확히 어느 함수에서 발생했는지 알 수 없으나, `kernel_oops:BUG:` 패턴이 존재한다는 점에서 커널 내부 오류가 배제되지 않음.

3. **부수적 요인**: **네트워크 활동/잠재적 공격**이 프로세스 비정상 행동을 유발해 메모리 소모를 가속화했을 가능성.  
   - 외부 IP/포트 기록이 많지만, 직접적인因果関係를 입증할 스택 트레이스나 프로세스 이름이 없으므로 confidence는 낮음.

### 권고 조치 (분석 기반)
| 조치 | 이유 |
|------|------|
| **메모리 하드웨어 진단** (MemTest86+, vendor 진단 툴) 실행 | `alloc magic is broken` 및 PXE‑E20 에러는 하드웨어 결함의典型적 신호 |
| **BIOS/UEFI 펌웨어 업데이트** 및 메모리 타이밍/전압 설정 검토 | PXE‑E20는 BIOS 단계에서 메모리 복사 오류를 나타냄 |
| **커널 로그(dmesg) 및 크래시 덤프 재수집** (가능하면 vmlinux 확보) | 정확한 오ops 지점 및 호출 스택을 얻어 커널 버그 여부 확인 |
| **프로세스별 메모리 사용량 모니터링** (top, ps, /proc/<pid>/smaps) | `out of memory`가 특정 프로세스 누수 때문인지 확인 |
| **보안 로그/IDS 검토** (포트 스캔, 비정상 연결) | 외부 IP/포트 기록이 많은 점을 고려해 악성 트래픽 여부 점검 |

> **요약**: 덤프 내부의 문자열과 패턴은 **메모리 하드웨어 손상**이 가장 plausibile한 root cause이며,これに続く 가능성으로 **커널 내부 버그** 또는 **과도한 메모리 소모**(누수/공격)가 있다. 신뢰도는 위 표와 같이 구분했으며, 추가적인 심볼 정보나 세부 스택 트레이스가 확보될 경우 정확도를 높일 수 있다.

In [ ]:
%%ai nemo -f markdown
아래 Android ramdump 분석 결과에서 핵심 이상 징후를 정리하고 가장 가능성 높은 root cause를 제안해줘.

{context_summary}

## Step 4. 대화형 질의

두 가지 방식으로 후속 질문이 가능합니다:

**방식 A — 아래 ipywidgets 채팅 UI** (이 노트북 안에서 LLMAssistant 사용)

**방식 B — 우측 Chat UI에서 `@Jupyternaut` 사용**
```
@Jupyternaut 방금 분석한 dump의 process 목록 중 의심스러운 것 알려줘
@Jupyternaut @file:notebooks/interactive_log_analyzer.ipynb 분석 결과 요약해줘
```
Jupyternaut는 이 노트북이 열려있으면 셀 내용을 직접 읽을 수 있습니다 (`get_active_cell_id` 도구 사용).

In [5]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# 분석 결과를 채팅 컨텍스트로 사전 주입
context_summary = json.dumps(
    {k: v for k, v in summary.items() if k != 'raw_chunk_samples'},
    ensure_ascii=False
)
assistant.reset()
assistant.history.append({
    'role': 'user',
    'content': f'다음은 memory.vmem 분석 결과입니다. 이 컨텍스트를 바탕으로 이후 질문에 답해줘:\n{context_summary}'
})
assistant.history.append({
    'role': 'assistant',
    'content': initial_analysis
})

# UI 구성
chat_log = widgets.Output(layout=widgets.Layout(height='400px', overflow_y='auto', border='1px solid #313244', padding='8px'))
input_box = widgets.Textarea(
    placeholder='질문을 입력하세요 (예: alloc magic broken이 의미하는게 뭐야?)',
    layout=widgets.Layout(width='100%', height='80px')
)
send_btn = widgets.Button(description='전송', button_style='primary', layout=widgets.Layout(width='80px'))
clear_btn = widgets.Button(description='대화 초기화', button_style='warning', layout=widgets.Layout(width='100px'))
status = widgets.Label(value='')

def on_send(b):
    question = input_box.value.strip()
    if not question:
        return
    input_box.value = ''
    send_btn.disabled = True
    status.value = '응답 대기 중...'
    with chat_log:
        display(Markdown(f'**You**: {question}'))
    response = assistant.chat(question)
    with chat_log:
        display(Markdown(f'**LLM**: {response}'))
        display(Markdown('---'))
    send_btn.disabled = False
    status.value = f'대화 턴: {len([m for m in assistant.history if m["role"]=="user"])}'

def on_clear(b):
    assistant.reset()
    assistant.history.append({'role': 'user', 'content': f'분석 컨텍스트:\n{context_summary}'})
    assistant.history.append({'role': 'assistant', 'content': initial_analysis})
    with chat_log:
        clear_output()
    status.value = '대화 초기화됨'

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

display(widgets.VBox([
    chat_log,
    input_box,
    widgets.HBox([send_btn, clear_btn, status])
]))